In [ ]:
import os
import re
import gc
import json
import random
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast
import networkx as nx
from scipy import special
from PIL import Image
from datasets import load_dataset
from itertools import islice

from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    CLIPTokenizer,
    CLIPModel,
    AutoModelForCausalLM
)




In [ ]:
torch.backends.cudnn.benchmark = True  # speed up convs on GPU

os.environ["HF_HOME"] = "..."
os.environ["HF_DATASETS_CACHE"] = "..."
os.environ["HF_HUB_CACHE"] = "..."
os.environ["TRANSFORMERS_CACHE"] = "..."

os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)



dataset = load_dataset(
    "MMInstruction/ArxivCap",
    split="train",
    streaming=True
)
samples = list(islice(dataset, 100))

In [ ]:
# Build figure-caption pairs for LLM experiments (keep PIL images, no re-encoding)

pairs = []

for s in samples:  # list of streamed papers you already collected
    arxiv_id = s.get("arxiv_id", "")
    title = s.get("title", "")
    abstract = s.get("abstract", "")

    for item in s.get("caption_images", []):
        parent_caption = (item.get("caption") or "").strip()

        for cp in item.get("cil_pairs", []):
            img = cp.get("image", None)  # PIL image
            if img is None:
                continue

            sub_caption = (cp.get("sub_caption") or "").strip()
            caption = sub_caption if sub_caption else parent_caption
            if not caption:
                continue

            pairs.append({
                "arxiv_id": arxiv_id,
                "title": title,
                "abstract": abstract,
                "image_file": cp.get("image_file", ""),
                "caption": caption,
                "image": img,  # <-- PIL Image, ready for LLaVA/Gemma3 processors
                "image_ocr": cp.get("image_ocr", None),  # optional, may help prompts
            })

print("Num (image, caption) pairs:", len(pairs))
print("Example keys:", pairs[0].keys() if pairs else "No pairs found")


In [ ]:
# =========================
# Utility
# =========================
def normalize_generation(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def _non_empty(s):
    return isinstance(s, str) and s.strip() != ""


# =========================
# vMF stats (stable)
# =========================
def _A_p_stable(kappa, d):
    if kappa <= 1e-8:
        return kappa / d
    if kappa >= 1e4:
        return 1.0 - (d - 1) / (2.0 * kappa) + (d - 1) * (d - 3) / (4.0 * kappa**2)
    num = special.ive(d / 2.0, kappa)
    den = special.ive(d / 2.0 - 1.0, kappa)
    return num / (den + 1e-300)


def _dA_p_from_A(kappa, A, d):
    if kappa <= 0:
        return 1.0 - A * A
    return 1.0 - A * A - ((d - 1) / kappa) * A


def _kappa_initial(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-12, 1 - 1e-12))
    return max((Rbar * (d - Rbar**2)) / (1.0 - Rbar**2), 1e-6)


def _kappa_upper_bound(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-9, 1 - 1e-9))
    return max((d - 1) / (2.0 * (1.0 - Rbar)), 10.0)


def vmf_stats(embeddings, newton_iters=8, tol=1e-7, use_bisection_backup=True):
    X = np.asarray(embeddings, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = X / norms

    N, d = X.shape
    R = X.sum(axis=0)
    R_norm = float(np.linalg.norm(R))
    if R_norm < 1e-12:
        return {"kappa": 0.0, "trace_var": 1.0, "var_parallel": 1.0 / d, "var_perp": 1.0 / d}

    Rbar = R_norm / max(N, 1)

    kappa = _kappa_initial(Rbar, d)
    ok_newton = True
    for _ in range(newton_iters):
        A = _A_p_stable(kappa, d)
        f = A - Rbar
        if abs(f) < tol:
            break
        dA = _dA_p_from_A(kappa, A, d)
        step = f / (dA + 1e-18)
        kappa_new = kappa - step
        if (not np.isfinite(kappa_new)) or (kappa_new <= 0) or (abs(step) > 0.5 * max(1.0, kappa)):
            ok_newton = False
            break
        kappa = kappa_new

    if (not ok_newton) and use_bisection_backup:
        lo = 0.0
        hi = _kappa_upper_bound(Rbar, d)
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            A_mid = _A_p_stable(mid, d)
            if A_mid < Rbar:
                lo = mid
            else:
                hi = mid
            if (hi - lo) < 1e-7 * (1.0 + mid):
                break
        kappa = 0.5 * (lo + hi)

    A = float(Rbar)
    var_perp = float(max(A / (kappa + 1e-18), 0.0))
    var_parallel = float(max(1.0 - ((d - 1.0) * A / (kappa + 1e-18)) - A * A, 0.0))
    trace_var = float(max(1.0 - A * A, 0.0))

    return {"kappa": float(kappa), "trace_var": trace_var, "var_parallel": var_parallel, "var_perp": var_perp}


# =========================
# DeBERTa semantic entropy (pairwise entailment clustering)
# =========================
class DebertaSemanticEntropy:
    def __init__(self, cache_dir: str, device: torch.device, hf_token: Optional[str] = None):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(
            "microsoft/deberta-large-mnli",
            cache_dir=cache_dir,
            token=hf_token
        )
        self.model = AutoModelForSequenceClassification.from_pretrained(
            "microsoft/deberta-large-mnli",
            cache_dir=cache_dir,
            token=hf_token
        ).to(device)
        self.model.eval()

    def _entail_prob(self, premise: str, hypothesis: str) -> float:
        inputs = self.tok(
            premise,
            hypothesis,
            return_tensors="pt",
            truncation=True
        ).to(self.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = F.softmax(logits, dim=1)
        # index 2 = entailment for DeBERTa-MNLI
        return probs[0][2].item()

    def score(self, gens: List[str], threshold=0.8) -> float:
        n = len(gens)
        if n == 0:
            return float("nan")

        G = nx.Graph()
        G.add_nodes_from(range(n))

        for i in range(n):
            for j in range(i + 1, n):
                e1 = self._entail_prob(gens[i], gens[j])
                e2 = self._entail_prob(gens[j], gens[i])
                if e1 > threshold and e2 > threshold:
                    G.add_edge(i, j)

        comps = list(nx.connected_components(G))

        labels = np.zeros(n, dtype=int)
        for cid, nodes in enumerate(comps):
            for node in nodes:
                labels[node] = cid

        counts = np.bincount(labels).astype(float)
        if counts.sum() <= 0:
            return float("nan")

        p = counts / counts.sum()
        return float(-np.sum(p * np.log(p + 1e-10)))


# =========================
# CLIP embedder (for DCU embeddings)
# =========================
class ClipTextEmbedder:
    def __init__(self, model_name: str, cache_dir: str, device: str, hf_token: Optional[str] = None):
        self.device = torch.device(device)
        self.tok = CLIPTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=hf_token)
        self.model = CLIPModel.from_pretrained(model_name, cache_dir=cache_dir, token=hf_token).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i: i + batch_size]
            inputs = self.tok(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)

            if self.device.type == "cuda":
                with autocast(dtype=torch.float16):
                    feats = self.model.get_text_features(**inputs)
            else:
                feats = self.model.get_text_features(**inputs)

            feats = F.normalize(feats, p=2, dim=1)
            embs.append(feats.cpu())
        return torch.cat(embs, dim=0).numpy()


# =========================
# ArxivCap captioning prompt (Title + Abstract context)
# =========================
def arxivcap_caption_prompt(row: pd.Series, max_ctx_chars: int = 1200) -> str:
    title = (row.get("title") or "").strip()
    abstract = (row.get("abstract") or "").strip()
    if len(abstract) > max_ctx_chars:
        abstract = abstract[:max_ctx_chars]

    return (
        "You are given a figure from an academic paper.\n"
        "Your task is to write the figure caption for the given image suitable for an academic paper.\n\n"
        "You may use the paper title and abstract as context.\n"
        "Do not mention the title or abstract explicitly.\n"
        "Do not invent information not supported by the image or context.\n\n"
        f"Paper title:\n{title}\n\n"
        f"Paper abstract:\n{abstract}\n\n"
        "Write the figure caption below. The output should have ONLY the caption and no other form text or information.\n"
    )


In [ ]:
# =========================
# Model loaders
# =========================
def load_magma(model_id: str, hf_token: Optional[str], cache_dir: str, device: torch.device):
    processor = AutoProcessor.from_pretrained(model_id, token=hf_token, cache_dir=cache_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        cache_dir=cache_dir,
        trust_remote_code=True,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)
    model.eval()
    return processor, model




# =========================
# Generation helpers
# =========================
def _repeat_batch(t: torch.Tensor, bs: int) -> torch.Tensor:
    return t.repeat(bs, *([1] * (t.ndim - 1)))


def generate_completions(
    processor,
    model,
    device: torch.device,
    prompt: str,
    image: Optional[Image.Image],
    *,
    n: int,
    batch_size: int,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
) -> List[str]:
    gens: List[str] = []

    # Build chat messages
    if image is not None:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
    else:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                ],
            }
        ]

    chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

    inputs = processor(text=chat_prompt, images=image, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    for start in range(0, n, batch_size):
        bs = min(batch_size, n - start)

        with torch.no_grad():
            if device.type == "cuda":
                with autocast(dtype=torch.float16):
                    out = model.generate(
                        **{k: _repeat_batch(v, bs) for k, v in inputs.items()},
                        do_sample=True,
                        temperature=temperature,
                        top_p=top_p,
                        max_new_tokens=max_new_tokens,
                        pad_token_id=processor.tokenizer.pad_token_id
                        if getattr(processor, "tokenizer", None) is not None
                        and processor.tokenizer.pad_token_id is not None
                        else processor.tokenizer.eos_token_id
                        if getattr(processor, "tokenizer", None) is not None
                        else None,
                    )
            else:
                out = model.generate(
                    **{k: _repeat_batch(v, bs) for k, v in inputs.items()},
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    max_new_tokens=max_new_tokens,
                    pad_token_id=processor.tokenizer.pad_token_id
                    if getattr(processor, "tokenizer", None) is not None
                    and processor.tokenizer.pad_token_id is not None
                    else processor.tokenizer.eos_token_id
                    if getattr(processor, "tokenizer", None) is not None
                    else None,
                )

        for i in range(bs):
            gen_ids = out[i][prompt_len:]
            if getattr(processor, "tokenizer", None) is not None:
                gen_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            else:
                gen_text = processor.decode(gen_ids, skip_special_tokens=True).strip()
            gens.append(gen_text)

        clear_gpu()

    return gens


# =========================
# Saving helper (JSONL is safest because "generations" is a list)
# =========================
def save_jsonl(rows: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Saved:", path, "| rows:", len(rows))

 

In [ ]:
# =========================
# Experiment runner (one model) -- ArxivCap captioning
# =========================
def run_experiment_for_model(
    df: pd.DataFrame,
    *,
    model_id: str,
    backend: str,  # "llava" | "gemma3"
    hf_token: Optional[str],
    cache_dir: str,
    clip_embedder: ClipTextEmbedder,
    se: DebertaSemanticEntropy,
    out_jsonl: str,
    n_generations: int = 10,
    batch_size: int = 1,
    max_new_tokens: int = 128,
    temperature: float = 0.7,
    top_p: float = 0.9,
    entail_threshold: float = 0.8,
) -> pd.DataFrame:
    print(f"\n=== Running ArxivCap captioning for {model_id} (backend={backend}) ===")

    device = se.device  # use same primary device

    
    processor, vlm = load_magma(model_id, hf_token, cache_dir, device)


    out_rows: List[Dict[str, Any]] = []

    for i, row in tqdm(df.reset_index(drop=True).iterrows(), total=len(df)):
        prompt = arxivcap_caption_prompt(row)
        image = row.get("image", None)  # PIL image

        gens = generate_completions(
            processor,
            vlm,
            device,
            prompt,
            image,
            n=n_generations,
            batch_size=batch_size,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        # --- vMF stats over CLIP embeddings of normalized generations ---
        cleaned = [normalize_generation(g) for g in gens if isinstance(g, str) and g.strip()]
        if len(cleaned) == 0:
            stats = {"kappa": np.nan, "trace_var": np.nan, "var_parallel": np.nan, "var_perp": np.nan}
        else:
            emb = clip_embedder.encode(cleaned, batch_size=32)
            stats = vmf_stats(emb)

        # --- semantic entropy (DeBERTa entailment clustering) ---
        try:
            sem_ent = se.score(gens, threshold=entail_threshold)
        except Exception:
            sem_ent = np.nan

        out_row = {
            "model_id": model_id,
            "backend": backend,
            "row_id": int(i),
            "n_generations": int(n_generations),
            "generations": gens,
            "semantic_entropy": float(sem_ent) if np.isfinite(sem_ent) else None,
            "kappa": stats["kappa"],
            "trace_var": stats["trace_var"],
            "var_parallel": stats["var_parallel"],
            "var_perp": stats["var_perp"],

            # --- keep ArxivCap metadata/context + GT caption ---
            "arxiv_id": row.get("arxiv_id", ""),
            "image_file": row.get("image_file", ""),
            "title": row.get("title", ""),
            "abstract": row.get("abstract", ""),
            "reference_caption": row.get("reference_caption", ""),
            "caption_used_in_prompt": None,  # for clarity: we did NOT include GT caption in prompt
        }

        out_rows.append(out_row)

    # Save JSONL (recommended)
    save_jsonl(out_rows, out_jsonl)

    out_df = pd.DataFrame(out_rows)

    # Cleanup big model to free GPU
    del vlm, processor
    clear_gpu()

    return out_df




In [ ]:
os.environ["HF_TOKEN"] = "..."
if __name__ == "__main__":
    # ---- You said: data is already available ----
    # Expect either:
    #   (A) `pairs` = list of dicts with keys:
    #       ["arxiv_id","title","abstract","image_file","caption","image", ...]
    #   OR
    #   (B) `df_arxivcap` already created.
    #
    # If you built `pairs` earlier using our extraction:
    #   caption field there is the ground-truth caption -> rename to reference_caption.

    ROOT_OUT = "results"
    CACHE_DIR = "model_cache"

    # HF token: set in your environment as HF_TOKEN, or set here.
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

    CLIP_ID = "openai/clip-vit-large-patch14"
    if "df_arxivcap" not in globals():
        if "pairs" not in globals():
            raise RuntimeError(
                "I don't see `pairs` or `df_arxivcap` in memory.\n"
                "Please run your ArxivCap streaming extraction first to create `pairs` "
                "(with PIL images), then re-run this block."
            )
        df_arxivcap = pd.DataFrame(pairs).copy()

# Ensure required columns exist
    needed = ["image", "title", "abstract", "caption"]
    missing = [c for c in needed if c not in df_arxivcap.columns]
    if missing:
        raise ValueError(f"df_arxivcap missing required columns: {missing}")

    print("Built ArxivCap df:", df_arxivcap.shape)
    print("Fraction with image:", df_arxivcap["image"].notna().mean())

    #df_debug = df_arxivcap.head(5).copy()    
    # ---- Shared components: CLIP + DeBERTa ----
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_embedder = ClipTextEmbedder(
        CLIP_ID,
        CACHE_DIR,
        device="cuda" if device.type == "cuda" else "cpu",
        hf_token=HF_TOKEN,
    )
    se = DebertaSemanticEntropy(cache_dir=CACHE_DIR, device=device, hf_token=HF_TOKEN)

    # ---- Models to run ----
    MODELS: List[Tuple[str, str, str]] = [
        ("microsoft/Magma-8B", "magma", os.path.join(ROOT_OUT, "arxivcap_magma_test.jsonl"))
    ]

    # ---- Run ----
    for model_id, backend, out_jsonl in MODELS:
        run_experiment_for_model(
            df_arxivcap,
            model_id=model_id,
            backend=backend,
            hf_token=HF_TOKEN,
            cache_dir=CACHE_DIR,
            clip_embedder=clip_embedder,
            se=se,
            out_jsonl=out_jsonl,
            n_generations=10,
            batch_size=1,
            max_new_tokens=128,     # captioning: shorter is better/faster
            temperature=0.7,
            top_p=0.9,
            entail_threshold=0.8,
        )

